# Cardio Catch Disease — 00. Business Understanding

> **Hands-On ML principle (Ch. 2):** Before touching any data, clearly frame the problem, define the objective, pick the performance measure, and understand how the solution will be used in production.

---

## 1. Business Context

**Cardio Catch Disease** is a healthcare company specialised in detecting cardiovascular diseases at early stages.

**Current situation:**
- Diagnosis is performed manually by specialist physicians
- Manual precision ranges between **55 % and 65 %** depending on team complexity
- Each diagnosis costs **\$1,000** (flat fee per patient)
- The company charges a **percentage of the fee** based on achieved precision:

| Accuracy band | Fee charged | Revenue / patient |
|---|---|---|
| 50 – 55 % | 5 % | \$50 |
| 55 – 60 % | 10 % | \$100 |
| 60 – 65 % | 15 % | \$150 |
| 65 – 70 % | 20 % | \$200 |
| 70 – 75 % | 25 % | \$250 |

**A 10-percentage-point improvement doubles revenue per patient.**

## 2. Business Goal

**Build an ML model that predicts cardiovascular risk from simple clinical variables, achieving higher accuracy than the current manual process.**

**Why ML is the right tool here:**
- We have 70,000 labelled historical patient records
- The task is well-defined: binary classification (disease / no disease)
- Output must be a **probability score** (enables risk ranking and threshold tuning)
- Needs both batch predictions and real-time API serving

**ML system type (Géron's taxonomy):**
- **Supervised learning** — labelled examples available
- **Binary classification** — predict presence / absence of disease
- **Model-based** — generalise from learned parameters, not instance memory
- **Batch training + online inference** — retrain periodically, serve predictions on demand

## 3. Performance Measure

> **Book principle:** Choose the metric before looking at the data to avoid unconsciously picking one that flatters your early models.

**Primary metric: ROC-AUC**
- Measures the model's ability to rank positive patients above negative ones
- Threshold-independent — lets us tune the operating point for different clinical capacities
- Standard benchmark in medical binary classification

**Secondary metrics:**

| Metric | Why it matters |
|---|---|
| **Recall** | Minimise false negatives (missed disease = delayed treatment, risk to life) |
| **Precision** | Minimise false positives (unnecessary referrals waste resources) |
| **F1-Score** | Harmonic balance of precision and recall |
| **Accuracy** | Maps directly to the company's revenue tier table |

## 4. Business Assumptions

| Assumption | Rationale |
|---|---|
| Recall prioritised over precision | False negatives delay treatment and risk lives |
| Blood pressure, age, BMI are strong signals | Supported by domain literature |
| Model supports — does not replace — physicians | Liability and clinical protocol requirements |
| Outliers in blood pressure are measurement errors | Physiologically impossible values (ap_hi < 0, ap_hi > 250) |
| Dataset is representative of the clinic population | Assumption needed for generalisation |

## 5. Dataset Overview

**Source:** [Kaggle — Cardiovascular Disease Dataset](https://www.kaggle.com/datasets/sulianova/cardiovascular-disease-dataset)

| Feature | Type | Description |
|---|---|---|
| `age` | int | Age in **days** (convert to years) |
| `gender` | cat | 1 = female, 2 = male |
| `height` | int | Height in cm |
| `weight` | float | Weight in kg |
| `ap_hi` | int | Systolic blood pressure (mmHg) |
| `ap_lo` | int | Diastolic blood pressure (mmHg) |
| `cholesterol` | cat | 1 = normal · 2 = above normal · 3 = well above |
| `gluc` | cat | 1 = normal · 2 = above normal · 3 = well above |
| `smoke` | bin | Smoker flag |
| `alco` | bin | Alcohol intake flag |
| `active` | bin | Physically active flag |
| `cardio` | bin | **Target**: 1 = cardiovascular disease |

**Engineered features (created in Notebook 03):**
- `age_years` = age / 365.25
- `bmi` = weight / (height_m)²
- `pulse_pressure` = ap_hi − ap_lo
- `high_blood_pressure` = 1 if ap_hi ≥ 140 or ap_lo ≥ 90

## 6. Solution Strategy (10 Steps)

Following the end-to-end ML project checklist from *Hands-On Machine Learning with Scikit-Learn and PyTorch* (Géron, 2025):

| # | Step | Notebook |
|---|---|---|
| 1 | **Data Description** — schema, shape, nulls, types | 01 |
| 2 | **Feature Engineering** — BMI, age_years, blood pressure features | 03 |
| 3 | **Data Filtering** — remove physiologically impossible values | 03 |
| 4 | **EDA** — validate hypotheses, find patterns | 02 |
| 5 | **Data Preparation** — Pipeline: impute → scale → encode | 03 |
| 6 | **Feature Selection** — separate IDs, target, inputs | 03 |
| 7 | **ML Modelling** — train & cross-validate LR vs RF vs XGBoost | 04 |
| 8 | **Hyperparameter Tuning** — RandomizedSearchCV on best model | 04 |
| 9 | **Business Translation** — accuracy tier → revenue impact | 04 |
| 10 | **Deployment** — FastAPI + Streamlit + Docker | 05 |

In [ ]:
# Revenue impact simulation
import matplotlib.pyplot as plt
import numpy as np

accuracy_bands   = ["50-55%", "55-60%", "60-65%", "65-70%", "70-75%"]
revenue_per_pt   = [50, 100, 150, 200, 250]
n_patients_month = 1_000

colors = ["#c0392b", "#e67e22", "#f1c40f", "#2ecc71", "#27ae60"]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(accuracy_bands, [r * n_patients_month / 1e3 for r in revenue_per_pt],
              color=colors, edgecolor="white", linewidth=1.2)

ax.axhline(y=150, color="gray", linestyle="--", alpha=0.7, label="Current baseline (≈65% accuracy)")
ax.set_xlabel("Model Accuracy Band", fontsize=12)
ax.set_ylabel("Monthly Revenue (thousands USD)", fontsize=12)
ax.set_title("Revenue Impact by Accuracy Band — 1,000 patients/month", fontsize=13, fontweight="bold")
ax.legend()
for bar, rev in zip(bars, revenue_per_pt):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f"\${rev}/pt", ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.tight_layout()
import os; os.makedirs("../reports/figures", exist_ok=True)
plt.savefig("../reports/figures/00_revenue_impact.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Summary

- **Problem type:** Supervised binary classification
- **Primary metric:** ROC-AUC (threshold-independent ranking)
- **Baseline to beat:** 65 % accuracy (current specialist)
- **Business KPI:** Each 5 pp accuracy improvement → next revenue tier
- **Delivery:** FastAPI endpoint + Streamlit UI + Docker for reproducibility

**Next → Notebook 01: Data Understanding**